In [0]:
/*
=============================================================================
PURPOSE: CAPM Scenario Analysis Matrix
=============================================================================
This view creates a comprehensive Cartesian product of all possible combinations
of dates, treasury maturities, beta coefficients, and market risk premiums
for Capital Asset Pricing Model (CAPM) calculations and scenario analysis.

TARGET SCHEMA: thekitchen.t (transformed data layer)
SOURCE VIEWS: 
- t.us_treasury_rates_tmp1 (risk-free rates by date and maturity)
- t.marketbeta_tmp1 (beta coefficients for different risk profiles)
- t.marketriskpremium_tmp1 (market risk premium scenarios)

BUSINESS VALUE:
- Enables exhaustive "what-if" scenario modeling for portfolio returns
- Pre-computes all possible CAPM input combinations for fast queries
- Supports backtesting different risk profiles across historical periods
- Facilitates sensitivity analysis: "How would returns change if beta = 1.3?"

CALCULATION FOUNDATION:
Expected Return = Risk-Free Rate + (Beta × Market Risk Premium)
- This view provides all three inputs for downstream CAPM calculations
- Each row represents one unique scenario (date × maturity × beta × MRP)

DATA VOLUME:
Number of rows = (# Dates) × (# Maturities) × (# Betas) × (# MRPs)
Example: 16,000 days × 3 maturities × 7 betas × 7 MRPs = ~2.4M rows

NOTE: TMP2 (temporary/intermediate) - typically used as input to final
      fact tables or aggregation views that calculate actual expected returns
=============================================================================
*/

CREATE OR REPLACE VIEW t.us_treasury_rates_tmp2 AS
-- ============================================
-- CAPM Input Combination Matrix
-- ============================================
-- Creates all possible combinations for comprehensive scenario analysis
SELECT
  f.Date AS Date,                      -- Observation date for risk-free rate
  f.MaturityKey AS MaturityKey,        -- Treasury maturity identifier (e.g., "10Y-US")
  f.Yield AS RiskFreeRate,             -- Risk-free rate (normalized as decimal)
  b.BetaKey AS BetaKey,                -- Beta category identifier (e.g., "BETA_100")
  m.MrpKey AS MrpKey                   -- Market risk premium scenario key
FROM
  t.us_treasury_rates_tmp1 AS f        -- Historical treasury yields (fact)
    CROSS JOIN t.marketbeta_tmp1 AS b  -- All beta coefficients (dimension)
    CROSS JOIN t.marketriskpremium_tmp1 AS m  -- All MRP scenarios (dimension)
ORDER BY
  f.Date DESC;                          -- Most recent dates first

/*
=============================================================================
USAGE EXAMPLES:
=============================================================================

-- Example 1: Calculate expected returns for all scenarios on latest date
WITH latest_scenarios AS (
  SELECT
    s.*,
    beta.BetaValue,
    beta.BetaLabel,
    mrp.MrpValue,
    mrp.MrpScenario
  FROM t.us_treasury_rates_tmp2 s
  JOIN t.marketbeta_tmp1 beta ON s.BetaKey = beta.BetaKey
  JOIN t.marketriskpremium_tmp1 mrp ON s.MrpKey = mrp.MrpKey
  WHERE s.Date = (SELECT MAX(Date) FROM t.us_treasury_rates_tmp2)
    AND s.MaturityKey = '10Y-US'  -- Use 10Y treasury as risk-free rate
)
SELECT
  BetaLabel,
  MrpScenario,
  ROUND(RiskFreeRate * 100, 2) AS risk_free_rate_pct,
  ROUND(MrpValue * 100, 2) AS mrp_pct,
  ROUND((RiskFreeRate + (BetaValue * MrpValue)) * 100, 2) AS expected_return_pct
FROM latest_scenarios
ORDER BY BetaValue, MrpValue;

-- Example 2: Sensitivity analysis - impact of changing beta
-- "If I keep MRP constant, how does expected return vary with beta?"
SELECT
  beta.BetaLabel,
  beta.BetaValue,
  ROUND(s.RiskFreeRate * 100, 2) AS risk_free_rate_pct,
  ROUND((s.RiskFreeRate + (beta.BetaValue * mrp.MrpValue)) * 100, 2) AS expected_return_pct,
  ROUND(beta.BetaValue * mrp.MrpValue * 100, 2) AS risk_premium_pct
FROM t.us_treasury_rates_tmp2 s
JOIN t.marketbeta_tmp1 beta ON s.BetaKey = beta.BetaKey
JOIN t.marketriskpremium_tmp1 mrp ON s.MrpKey = mrp.MrpKey
WHERE s.Date = (SELECT MAX(Date) FROM t.us_treasury_rates_tmp2)
  AND s.MaturityKey = '10Y-US'           -- Fixed: 10Y Treasury
  AND mrp.MrpScenario = 'Base'           -- Fixed: Base case MRP
ORDER BY beta.BetaValue;

-- Example 3: Historical backtest - which beta profile performed best?
WITH monthly_returns AS (
  SELECT
    DATE_TRUNC('month', s.Date) AS month,
    s.BetaKey,
    beta.BetaLabel,
    AVG(s.RiskFreeRate + (beta.BetaValue * mrp.MrpValue)) AS avg_expected_return
  FROM t.us_treasury_rates_tmp2 s
  JOIN t.marketbeta_tmp1 beta ON s.BetaKey = beta.BetaKey
  JOIN t.marketriskpremium_tmp1 mrp ON s.MrpKey = mrp.MrpKey
  WHERE s.MaturityKey = '10Y-US'
    AND s.Date >= '2020-01-01'
    AND mrp.MrpScenario = 'Base'  -- Use base case MRP
  GROUP BY 1, 2, 3
)
SELECT
  BetaLabel,
  ROUND(AVG(avg_expected_return) * 100, 2) AS avg_monthly_return_pct,
  ROUND(STDDEV(avg_expected_return) * 100, 2) AS return_volatility_pct,
  ROUND(AVG(avg_expected_return) / NULLIF(STDDEV(avg_expected_return), 0), 2) AS sharpe_ratio
FROM monthly_returns
GROUP BY BetaLabel, BetaKey
ORDER BY sharpe_ratio DESC;

-- Example 4: Compare risk-free rate choices (different maturities)
SELECT
  mat.Description AS maturity,
  AVG(s.RiskFreeRate) AS avg_risk_free_rate,
  ROUND(AVG(s.RiskFreeRate + (beta.BetaValue * mrp.MrpValue)) * 100, 2) AS avg_expected_return_pct
FROM t.us_treasury_rates_tmp2 s
JOIN t.marketbeta_tmp1 beta ON s.BetaKey = beta.BetaKey
JOIN t.marketriskpremium_tmp1 mrp ON s.MrpKey = mrp.MrpKey
JOIN t.maturity_tmp1 mat ON s.MaturityKey = mat.MaturityKey
WHERE s.Date >= '2020-01-01'
  AND beta.BetaKey = 'BETA_100'          -- Market beta
  AND mrp.MrpScenario = 'Base'           -- Base case MRP
GROUP BY mat.Description, mat.MaturityYears
ORDER BY mat.MaturityYears;
=============================================================================
*/